# Chapter 09 — Mixed Precision Training with PyTorch

What if you could train your model twice as fast and use half the memory — just by using different number formats? That's the promise of mixed-precision training. This chapter explores how reducing numerical precision from 32-bit to 16-bit (and beyond to 8-bit) accelerates training without sacrificing model quality.

### Glossary / Glossário

• FP32 — 32-bit floating point, full precision (4 bytes per number). / Ponto flutuante 32 bits, precisão total (4 bytes por número).
• FP16 — 16-bit floating point, half precision (2 bytes, faster but smaller range). / Ponto flutuante 16 bits, meia precisão (2 bytes, mais rápido mas range menor).
• BF16 — Brain Float 16, 16-bit with FP32's exponent range, stable for training. / Brain Float 16, 16 bits com range de expoente do FP32, estável para treinamento.
• FP8 — 8-bit floating point, ultra-low precision used primarily for inference. / Ponto flutuante 8 bits, precisão ultra-baixa usada primariamente para inferência.
• mixed precision — training with FP16/BF16 compute and FP32 master weights for speed + stability. / Treinamento com compute em FP16/BF16 e pesos mestres em FP32 para velocidade + estabilidade.
• GradScaler — PyTorch utility that scales loss to prevent FP16 gradient underflow. / Utilidade do PyTorch que escala o loss para prevenir underflow de gradientes em FP16.
• Tensor Cores — specialized GPU hardware for fast low-precision matrix math. / Hardware especializado da GPU para multiplicação de matrizes em baixa precisão.

In [ ]:
import torch

print("=== Floating Point Formats ===")
for dtype, name, bits in [
    (torch.float32, "FP32", 32),
    (torch.float16, "FP16", 16),
    (torch.bfloat16, "BF16", 16),
]:
    t = torch.tensor(3.141592653589793, dtype=dtype)
    print(f"{name} ({bits} bits): {t.item():.10f}  | max={torch.finfo(dtype).max:.2e}")

print("\n=== Mixed Precision with autocast ===")
a = torch.randn(64, 64)
b = torch.randn(64, 64)

c_fp32 = torch.mm(a, b)
print(f"Without autocast: input={a.dtype}, output={c_fp32.dtype}")

with torch.autocast('cpu', dtype=torch.bfloat16):
    c_bf16 = torch.mm(a, b)
print(f"With autocast:    input={a.dtype}, output={c_bf16.dtype}")

diff = (c_fp32 - c_bf16.float()).abs().mean()
print(f"Mean abs diff:    {diff.item():.6f}")

print("\n=== Memory Savings ===")
size = 4096
for dtype, name in [(torch.float32, "FP32"), (torch.bfloat16, "BF16")]:
    t = torch.randn(size, size, dtype=dtype)
    mb = t.element_size() * t.nelement() / 1e6
    print(f"{name} tensor ({size}x{size}): {mb:.1f} MB")

In [ ]:
import torch
from torch.cuda.amp import autocast, GradScaler

model = model.cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scaler = GradScaler()

for batch in dataloader:
    optimizer.zero_grad()
    with autocast(dtype=torch.bfloat16):
        logits = model(batch["input_ids"].cuda())
        loss = F.cross_entropy(logits.view(-1, vocab_size), batch["labels"].cuda().view(-1))
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

---

**Course:** Aprenda LLM | **Chapter 09**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.